## Environment Setup & Imports

If you are running this notebook on **Databricks**, uncomment and run the following line in the cell below to install the local `landseg` package in your cluster's Python environment:
```python
# %pip install -e ..
```

In [ ]:
# Databricks Setup

import os
import sys
sys.path.insert(0, os.path.join(os.path.abspath('..'), 'src'))
import landseg.adapters.api as api

# Check if running in a Databricks environment
IS_DATABRICKS = 'DATABRICKS_RUNTIME_VERSION' in os.environ

# Define the experiment root path where artifacts and results will be stored.
# Relative paths are standard for local repositories. On Databricks, consider utilizing
# absolute paths on a Unity Catalog Volume or DBFS path (e.g. '/Volumes/my_catalog/my_schema/my_volume/experiment/').
EXP_ROOT = '../experiment/'
# Define name of the dataset, used for naming outputs and logging
DATASET_NAME = 'demo_data'

if IS_DATABRICKS:
    # EXP_ROOT = '/Volumes/my_catalog/my_schema/my_volume/experiment/'
    print('Running on Databricks. For high performance, configure EXP_ROOT using a Unity Catalog Volume or DBFS path.')

## Configure Data Harmonization (ETL) and Run

In [ ]:
# NOTE
# This is a demo script to show how to configure the data harmonization pipeline
# The purpose is to reproject and stack raw rasters into canonical Virtual Rasters (.vrt)
# The output artifacts are saved under artifacts/harmonized/run_XXXX/
# Typically this is run once when raw rasters arrive or when adjusting spatial canvas properties
# Adjust the parameters and file paths as needed for your specific use case
# Relative file paths are used here for demonstration (data shipped with the codebase)
# Use absolute file paths to ensure consistency across different environments

# init configurator
configurator = api.DataHarmonizationConfigurator(EXP_ROOT, DATASET_NAME)

# set target spatial canvas (CRS, resolution, and optional extent reference)
configurator.set_canvas(
    # EPSG code for target spatial projection
    target_crs='EPSG:3161',
    # target spatial pixel size resolution (metres)
    target_resolution=20.0,
    # optional path to spatial extent reference raster
    reference_raster=os.path.join(EXP_ROOT, 'input/extent_reference/sample_extent.tif')
)

# set continuous feature input rasters (optical bands, DEMs, etc.)
configurator.set_features({
    'sentinel2': os.path.join(EXP_ROOT, 'input/raw/sample_sentinel2.tif'),
    'dem': os.path.join(EXP_ROOT, 'input/raw/sample_dem.tif')
})

# set categorical label rasters (nearest-neighbor resampled)
configurator.set_labels({
    'landcover': os.path.join(EXP_ROOT, 'input/raw/sample_landcover.tif')
})

# run data-harmonize pipeline
report = api.run(configurator.running_root_config)
print('ETL Report Summary:', report)